# Annadata — Crop Recommendation → ONNX

Uses Colab sklearn only. Installs: `kaggle skl2onnx onnx` (no TensorFlow).


In [ ]:
!pip install -q kaggle skl2onnx onnx onnxmltools
import json, os
from pathlib import Path
KAGGLE_USERNAME, KAGGLE_KEY = "aftaabkazi", "YOUR_KAGGLE_KEY_HERE"
home = Path.home() / ".kaggle"; home.mkdir(exist_ok=True)
(home / "kaggle.json").write_text(json.dumps({"username": KAGGLE_USERNAME, "key": KAGGLE_KEY}))
os.chmod(home / "kaggle.json", 0o600)
!kaggle datasets download -d atharvaingle/crop-recommendation-dataset -p /content --unzip
print("done")


In [ ]:
import json
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report
from sklearn.preprocessing import LabelEncoder
from skl2onnx import convert_sklearn
from skl2onnx.common.data_types import FloatTensorType

OUT = Path("/content/agrosight_exports"); OUT.mkdir(exist_ok=True)
csv = next(Path("/content").rglob("Crop_recommendation.csv"))
df = pd.read_csv(csv)
FEATURES = ["N", "P", "K", "temperature", "humidity", "ph", "rainfall"]
cols = {c.lower(): c for c in df.columns}
X = df[[cols[f.lower()] for f in FEATURES]].astype(np.float32)
le = LabelEncoder()
y = le.fit_transform(df[cols.get("label", df.columns[-1])].astype(str))
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=200, max_depth=12, random_state=42, n_jobs=-1)
rf.fit(Xtr, ytr)
acc = accuracy_score(yte, rf.predict(Xte))
print("hold-out accuracy", acc)
print(classification_report(yte, rf.predict(Xte), target_names=le.classes_))
onnx_model = convert_sklearn(rf, initial_types=[("float_input", FloatTensorType([None, 7]))], target_opset=12)
(OUT / "crop_rec.onnx").write_bytes(onnx_model.SerializeToString())
meta = {"features": FEATURES, "classes": list(le.classes_), "accuracy": round(float(acc)*100, 2)}
(OUT / "crop_rec_meta.json").write_text(json.dumps(meta, indent=2))
print(meta)
from google.colab import files
files.download(str(OUT / "crop_rec.onnx"))
files.download(str(OUT / "crop_rec_meta.json"))
